In [11]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd

from data import make_design_arrays, RuntimeDataset
from models import (
    model_offset_hf, model_offset_interaction_no_ntasks,
    fit_mcmc, flatten_posterior, save_fit, load_and_validate_fit,
    fit_anchored,
)
from energy import (
    run_energy_mc_hf, run_energy_mc_str_norm,
    decompose_variance_aggregate,
)

In [12]:
import numpyro
numpyro.set_host_device_count(4)

import jax
print(jax.local_device_count())

4


In [14]:
import os
print(os.getcwd())

/Users/dm/Documents/coding/thesis_code/model


In [15]:
df_train_hf = pd.read_csv("results/df_train_hf.csv")
df_train_sn = pd.read_csv("results/df_train_str_norm.csv")

corpora_hf, corp2idx_hf = make_design_arrays(df_train_hf)
corpora_sn, corp2idx_sn = make_design_arrays(df_train_sn)

hf_dataset = RuntimeDataset(df=df_train_hf, corpora=corpora_hf, corp2idx=corp2idx_hf, stage="heuristic_filtering")
sn_dataset = RuntimeDataset(df=df_train_sn, corpora=corpora_sn, corp2idx=corp2idx_sn, stage="string_normalisation")

print(hf_dataset)
print(sn_dataset)

RuntimeDataset(stage='heuristic_filtering', n_obs=95, n_corpora=6)
RuntimeDataset(stage='string_normalisation', n_obs=153, n_corpora=9)


In [ ]:
y_hf = hf_dataset.df["log_rt_per_char"].values
alpha_cfg_hf = hf_dataset.df["alpha_cfg"].values
J_hf = len(hf_dataset.corpora)

mcmc_hf, idata_hf = fit_mcmc(
    model_offset_hf, jax.random.PRNGKey(0),
    hf_dataset.c_idx, alpha_cfg_hf, J_hf, y=y_hf,
    progress_bar = True
)

save_fit(idata_hf, "results/fit_hf_production.nc", "model_offset_hf")

alpha_c_post_hf = flatten_posterior(idata_hf, "alpha_c")
beta_c_post_hf  = flatten_posterior(idata_hf, "beta_c")

print("alpha_c means:", dict(zip(hf_dataset.corpora, alpha_c_post_hf.mean(axis=0).round(3))))

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]





Running chain 0:   0%|          | 0/4000 [00:05<?, ?it/s]




Running chain 0:   5%|▌         | 200/4000 [00:07<00:39, 97.06it/s]

Running chain 0:  10%|█         | 400/4000 [00:08<00:30, 117.53it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:10<00:28, 121.09it/s]





Running chain 0:  25%|██▌       | 1000/4000 [00:14<00:31, 94.53it/s]


Running chain 0:  30%|███       | 1200/4000 [00:16<00:26, 105.78it/s]


Running chain 0:  40%|████      | 1600/4000 [00:18<00:18, 130.00it/s]


Running chain 0:  45%|████▌     | 1800/4000 [00:20<00:16, 136.94it/s]


Running chain 0:  55%|█████▌    | 2200/4000 [00:24<00:15, 114.19it/s]




Running chain 0:  60%|██████    | 2400/4000 [00:26<00:15, 103.05it/s]




Running chain 0:  65%|██████▌   | 2600/4000 [00:29<00:14, 93.80it/s] 


Running chain 0:  70%|███████   | 2800/4000 [00:30<00:11, 103.17it/s]

Running chain 0:  75%|███████▌  | 3000/4000 [00:32<00:10, 98.63it/s] 


Running chain 0

alpha_c means: {'am_stories_str_norm': np.float32(-11.856), 'cc_github_str_norm': np.float32(-14.868), 'cc_openalex_str_norm': np.float32(-12.514), 'kb_books_str_norm': np.float32(-12.336), 'naturalis_str_norm': np.float32(-12.521), 'woogle_str_norm': np.float32(-12.748)}


In [ ]:
y_sn = sn_dataset.df["log_rt_per_char"].values
form_idx_sn = sn_dataset.df["form_idx"].values
J_sn = len(sn_dataset.corpora)
K_sn = 3  # NFC, NFD, NFKC

mcmc_sn, idata_sn = fit_mcmc(
    model_offset_interaction_no_ntasks, jax.random.PRNGKey(0),
    sn_dataset.c_idx, form_idx_sn, J_sn, K_sn, y=y_sn,
    progress_bar = True
)

save_fit(idata_sn, "results/fit_str_norm_production.nc", "model_offset_interaction_no_ntasks")

alpha_c_post_sn = flatten_posterior(idata_sn, "alpha_c")
delta_corpus_post_sn = flatten_posterior(idata_sn, "delta_corpus")

print("alpha_c means:", dict(zip(sn_dataset.corpora, alpha_c_post_sn.mean(axis=0).round(3))))

Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]



  0%|          | 0/4000 [00:00<?, ?it/s]




Running chain 0:   0%|          | 0/4000 [00:05<?, ?it/s]





Running chain 0:   5%|▌         | 200/4000 [00:09<01:04, 58.60it/s]


Running chain 0:  10%|█         | 400/4000 [00:11<00:52, 68.04it/s]


Running chain 0:  15%|█▌        | 600/4000 [00:13<00:39, 85.89it/s]


Running chain 0:  20%|██        | 800/4000 [00:14<00:33, 96.70it/s]



Running chain 0:  25%|██▌       | 1000/4000 [00:16<00:28, 105.95it/s]

Running chain 0:  30%|███       | 1200/4000 [00:18<00:26, 106.30it/s]


Running chain 0:  35%|███▌      | 1400/4000 [00:20<00:23, 110.85it/s]



Running chain 0:  40%|████      | 1600/4000 [00:23<00:27, 87.54it/s] 


Running chain 0:  45%|████▌     | 1800/4000 [00:25<00:24, 88.93it/s]


Running chain 0:  50%|█████     | 2000/4000 [00:27<00:21, 91.34it/s]




Running chain 0:  55%|█████▌    | 2200/4000 [00:30<00:20, 86.56it/s]

Running chain 0:  60%|██████    | 2400/4000 [00:31<00:16, 96.56it/s]


Running chain 0:  65%|██████▌   | 2600

alpha_c means: {'am_stories': np.float32(-13.914), 'cc_github': np.float32(-14.065), 'cc_openalex': np.float32(-13.977), 'eu_parliament': np.float32(-13.851), 'kb_books': np.float32(-14.052), 'naturalis': np.float32(-14.05), 'rechtspraak': np.float32(-13.502), 'tweedekamer': np.float32(-14.15), 'woogle': np.float32(-14.193)}


In [ ]:
df_energy_lookup_hf = (
    df_train_hf.groupby(["corpus", "config", "alpha_cfg"])["total_chars"]
    .mean()
    .reset_index()
)

df_energy_hf = run_energy_mc_hf(
    df_energy_lookup_hf, alpha_c_post_hf, beta_c_post_hf, hf_dataset.corpora,
    p_task_mean=28.66, p_task_std=2.24,
)
df_energy_hf.to_csv("results/df_energy_hf_predicted.csv", index=False)

df_energy_lookup_sn = (
    df_train_sn.groupby(["corpus", "normalization_form"])["total_chars"]
    .mean()
    .reset_index()
)

P_TASK_MIXTURE = {
    0: {"means": [22.0, 29.0], "stds": [2.0, 2.0], "weights": [0.5, 0.5]},  # NFC
    1: {"means": [21.0, 28.0], "stds": [2.0, 2.0], "weights": [0.4, 0.6]},  # NFD
    2: {"means": [21.0, 29.0], "stds": [2.0, 2.0], "weights": [0.6, 0.4]},  # NFKC
}
FORM_MAP = {"NFC": 0, "NFD": 1, "NFKC": 2}

df_energy_sn = run_energy_mc_str_norm(
    df_energy_lookup_sn, alpha_c_post_sn, delta_corpus_post_sn,
    sn_dataset.corpora, FORM_MAP, P_TASK_MIXTURE,
)
df_energy_sn.to_csv("results/df_energy_str_norm_predicted.csv", index=False)

print(df_energy_hf[["corpus", "config", "E_mean_kJ"]].head())
print(df_energy_sn[["corpus", "normalization_form", "E_mean_kJ"]].head())

                corpus       config    E_mean_kJ
0  am_stories_str_norm  ro_alpha060  1356.444776
1  am_stories_str_norm  ro_alpha065   984.960350
2  am_stories_str_norm  ro_alpha075   965.191960
3  am_stories_str_norm  ro_alpha080  1295.071396
4   cc_github_str_norm  ro_alpha060    92.930652
       corpus normalization_form   E_mean_kJ
0  am_stories                NFC  185.554187
1  am_stories                NFD  184.699694
2  am_stories               NFKC  176.286168
3   cc_github                NFC  173.824697
4   cc_github                NFD  171.816377


In [9]:
result_hf = decompose_variance_aggregate(
    rt_samples_list=df_energy_hf["rt_samples"].tolist(),
    P_samples_list=df_energy_hf["P_samples"].tolist(),
)
print(f"Filtering: runtime={result_hf['share_runtime']:.1%}, hardware={result_hf['share_power']:.1%}")

result_sn = decompose_variance_aggregate(
    rt_samples_list=df_energy_sn["rt_samples"].tolist(),
    P_samples_list=df_energy_sn["P_samples"].tolist(),
)
print(f"Str norm: runtime={result_sn['share_runtime']:.1%}, hardware={result_sn['share_power']:.1%}")

Filtering: runtime=1.0%, hardware=99.0%
Str norm: runtime=0.1%, hardware=99.9%
